# 🚀 Large-Scale Cloud GPU Patent Embedder & FAISS Vector Indexer
### SDTD Multi-Agent Patent Intelligence System — Cloud Worker

This notebook offloads the heavy embedding of **tens of thousands of patents** to Google Colab's free GPU (T4/V100/A100).

**What this notebook does:**
1. Streams genuine USPTO patent applications from large-scale patent datasets (**10,000 to 100,000+ patents**).
2. Performs fast **FP16 batched GPU inference** with `anferico/bert-for-patents` (345M parameters).
3. Builds a normalized **FAISS Inner-Product (Cosine Similarity) index** (`patents_faiss.index`) and metadata table (`patents_meta.parquet`).
4. Automatically triggers browser download of both files so you can drop them into your local project.

⚡ **Important First Step**: Ensure GPU is active! Go to **Runtime → Change runtime type → Hardware accelerator → T4 GPU**.

In [ ]:
# Step 1: Check GPU & Install Dependencies
!nvidia-smi
!pip install -q transformers torch datasets pyarrow pandas tqdm faiss-cpu

In [ ]:
# Step 2: Storage Setup (Local Colab Storage or Google Drive)
import os

# Set to True if you want to also persist the outputs inside your Google Drive
MOUNT_GOOGLE_DRIVE = False

OUTPUT_DIR = "/content"
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = '/content/drive/MyDrive/patent_faiss_index'
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print(f"[*] Outputs will be saved to Google Drive: {OUTPUT_DIR}")
    except Exception as e:
        print(f"[*] Drive mount skipped ({e}). Using /content...")
        OUTPUT_DIR = "/content"
else:
    print(f"[*] Using fast Colab storage at: {OUTPUT_DIR}")

In [ ]:
#@title Step 3: Stream Real Large-Scale USPTO Patent Dataset { form-width: "45%" }
# Select how large you want your patent search database to be:
DATASET_SCALE = "50,000 Patents (Production Scale ~15 min)" #@param ["10,000 Patents (Fast ~3 min)", "25,000 Patents (~8 min)", "50,000 Patents (Production Scale ~15 min)", "100,000 Patents (Enterprise ~30 min)"]

scale_map = {
    "10,000 Patents (Fast ~3 min)": 10000,
    "25,000 Patents (~8 min)": 25000,
    "50,000 Patents (Production Scale ~15 min)": 50000,
    "100,000 Patents (Enterprise ~30 min)": 100000
}
MAX_PATENTS = scale_map.get(DATASET_SCALE, 50000)

import re
import ast
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset

print(f"Streaming real USPTO patent records (Target scale: {MAX_PATENTS:,} patents)...")

records = []
texts = []

# Primary large-scale patent stream
try:
    ds = load_dataset("ccdv/patent-classification", split="train", streaming=True)
    for item in tqdm(ds, total=MAX_PATENTS, desc="Streaming large patent corpus"):
        full_text = (item.get("text") or "").strip()
        label = item.get("label", 0)
        # Map label index to primary IPC sections (A-H)
        ipc_section = ["A", "B", "C", "D", "E", "F", "G", "H", "Y"][label % 9]
        # Extract first line as title and remainder as description
        parts = full_text.split("\n", 1)
        title = parts[0][:150].strip() or f"Patent Document {len(records)+1}"
        abstract = parts[1][:1200].strip() if len(parts) > 1 else full_text[:1200].strip()
        abstract = re.sub(r'<[^>]+>', ' ', abstract).strip()
        
        # Extract any specific 4-char IPC code in text, or use section default
        found_codes = re.findall(r'\b([A-HY][0-9]{2}[A-Z])\b', full_text[:2000])
        ipc_codes = list(dict.fromkeys(found_codes)) if found_codes else [f"{ipc_section}06F"]
        
        if len(abstract) > 30:
            meta_text = f"{title}. {abstract}"
            records.append({
                "patent_id": f"US_PAT_{len(records)+1:07d}",
                "meta": meta_text,
                "codes": ipc_codes,
            })
            texts.append(meta_text)
            
        if len(records) >= MAX_PATENTS:
            break
except Exception as e:
    print(f"Notice: fallback to HUPD stream ({e})")
    ds_fallback = load_dataset("late-interacters/hupd-2007-v2-small", split="train")
    for item in ds_fallback:
        title = (item.get("title") or "").strip()
        abstract = (item.get("abstract") or "").strip()
        if len(title) > 10 and len(abstract) > 30:
            meta_text = f"{title}. {abstract}"
            records.append({
                "patent_id": str(item.get("application_number", f"PAT_{len(records):06d}")),
                "meta": meta_text,
                "codes": ["G06F"],
            })
            texts.append(meta_text)
        if len(records) >= MAX_PATENTS:
            break

print(f"\nPrepared {len(records):,} clean USPTO patent documents ready for GPU embedding!")

In [ ]:
# Step 4: High-Speed Batch GPU Inference with Patent-BERT (FP16)
import torch
import numpy as np
from transformers import BertModel, BertTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Active compute device: {device}")
if device == "cuda":
    print(f"[*] GPU: {torch.cuda.get_device_name(0)}")

model_name = "anferico/bert-for-patents"
print(f"Loading tokenizer and weights for '{model_name}'...")
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name).to(device)

# Enable FP16 half-precision for 3x faster GPU throughput
if device == "cuda":
    model = model.half()
model.eval()

BATCH_SIZE = 128 if device == "cuda" else 16  # Large batches on Colab GPU
embeddings = []

print(f"Embedding {len(texts):,} patents in batches of {BATCH_SIZE} on Cloud GPU...")
with torch.no_grad():
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="BERT GPU Forward Pass"):
        batch = texts[i : i + BATCH_SIZE]
        tok = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)
        
        out = model(**tok)
        # Mean pooling across token representations
        pooled = torch.mean(out.last_hidden_state, dim=1)
        embeddings.append(pooled.float().cpu().numpy())

embeddings = np.vstack(embeddings).astype(np.float32)
print("\n[*] Embeddings generated successfully!")
print(f"[*] Matrix shape: {embeddings.shape} (Patents x 1024 Dim)")

In [ ]:
# Step 5: Build FAISS Vector Index and Save Artifacts
import faiss

# Normalize vectors for exact Cosine Similarity (Inner Product)
faiss.normalize_L2(embeddings)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

index_file = os.path.join(OUTPUT_DIR, "patents_faiss.index")
meta_file = os.path.join(OUTPUT_DIR, "patents_meta.parquet")

faiss.write_index(index, index_file)
pd.DataFrame(records).to_parquet(meta_file, index=False)

print("=" * 60)
print(f"✅ Successfully built FAISS index with {index.ntotal:,} patent vectors!")
print(f"1. Index path: {index_file} ({round(os.path.getsize(index_file)/(1024*1024), 2)} MB)")
print(f"2. Meta path:  {meta_file} ({round(os.path.getsize(meta_file)/(1024*1024), 2)} MB)")
print("=" * 60)

In [ ]:
# Step 6: Quick In-Colab Test Query
sample_query = "A semiconductor cooling system with phase change heat sink and micro-channels"
print(f"Testing retrieval with query: '{sample_query}'")

with torch.no_grad():
    q_tok = tokenizer([sample_query], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    q_emb = torch.mean(model(**q_tok).last_hidden_state, dim=1).float().cpu().numpy().astype(np.float32)
    faiss.normalize_L2(q_emb)

scores, indices = index.search(q_emb, 3)
print("\n--- Top 3 Retrieved Patents ---")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    rec = records[idx]
    print(f"#{rank} [Score: {score:.4f}] Patent ID: {rec['patent_id']} | IPC: {rec['codes']}")
    print(f"   {rec['meta'][:150]}...\n")

In [ ]:
# Step 7: Download Index & Metadata directly to your PC
from google.colab import files

print("Triggering browser download for both files...")
files.download(index_file)
files.download(meta_file)
print("\nDone! Move both downloaded files into your local project root folder:")
print("  -> patents_faiss.index")
print("  -> patents_meta.parquet")